In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient III: a learned critic

## Where this session fits

> **The limitation we start from: REINFORCE's weight is still one noisy sampled return, and with a single end reward every step gets the same weight**

- **The question:** can a learned estimate of the state value replace part of the sampled return?
    - so that the weight is less noisy and each step gets its own
- from B2: the weight on each score function is an advantage, the action value minus the state value
    - any baseline that ignores the action leaves the expected gradient unchanged (the zero-mean score lemma)
    - the state value is the natural baseline, and this session **learns** it with a network
- **Position: Part B (policy gradient), session 3 of its 4 sessions**
    - B1's slots: this session fills slot 1, the weight, with an advantage from a **critic**, and slot 3, the baseline

<img src="images/partB_arc_b3.png" width=70%>

- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say what a critic is: its input, its output, and what it is trained on
2. say the two roles of a critic: a baseline (no bias) and a bootstrap (bias)
3. say why the TD error is a sample of the advantage
4. read one actor-critic step, with its two updates
5. say what the dial between the return and the TD error trades, and place actor-critic in the slot table


# What a critic is (step 1 of 4)

**Why this section?** The word has appeared since B1, with only a gloss. First the idea, then the definition.

> **An actor-critic method has two networks. The actor is the policy and chooses actions. The critic predicts how well things will go from a state.**

<img src="images/b3_actor_critic_loop.png" width=62%>

- **two terms we will use from now on**
    - the **actor** is the policy $\pi_\theta$, with parameters $\theta$
    - the **critic** is a network $\statevalfun_\phi$, with parameters $\phi$, that estimates the value of a state
- the actor learns from whether the outcome beat the critic's expectation: the TD error $\delta$ (step 4)
- four steps: what a critic is, role 1, role 2, and both together (the TD error)


**The critic, precisely**

> **The critic $\statevalfun_\phi$ learns the state value: how much return to expect from a state, as in Part A.**

<img src="images/fl_value_map.png" width=28%>

- input: a state $\state$; output: one number, the expected return from $\state$
    - the $\pi$ in $\statevalfun_\pi$ is the current policy $\pi_\theta$, as in B2
    - on FrozenLake the critic would learn this map (the exact values of the uniform policy)
- parameters: $\phi$ (the actor's are $\theta$)
- trained by regression toward the target $\rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1})$, held fixed
    - the target is the number a value table would be updated toward (Part A), if there were a table
    - the network is moved toward it instead
    - the gap between target and prediction is the TD error (step 4)
- the critic is a *use* of Part A's value machinery, not a return to value-based control
    - the policy is still $\pi_\theta$, not an argmax over values


**Step 2 of 4, role 1: the critic as a baseline**

> **Subtract the critic's value from the sampled return: $b(\state) = \statevalfun_\phi(\state)$. It adds no bias, however wrong the critic is.**

- the weight becomes the sampled return minus the critic's value: $\rlret_{\tt} - \statevalfun_\phi(\stateseq_\tt)$
- **why no bias: B2 showed that any baseline $b(\state)$ that ignores the action adds none**
    - the critic's value is such a baseline: it depends on the state, not on the action
    - $\phi$ is held fixed while this step is sampled

- **what this gives:** the critic may be inaccurate, even bad, without biasing the actor
    - its accuracy changes only the variance


**Step 3 of 4, role 2: the critic as a bootstrap**

> **Replace the rest of the sampled return by the critic's estimate. The variance falls much further, and a bias appears.**

<img src="images/b3_bootstrap.png" width=56%>

- the return from step $\tt$ is replaced by $\rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1})$: one reward, then the critic's value of the next state, as TD replaced Monte Carlo
    - with the baseline of role 1 as well, it is $\rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1}) - \statevalfun_\phi(\stateseq_\tt)$
    - step 4 names it: the TD error
- the price: **bias**
    - an inaccurate critic biases the weight (the bias of bootstrapping, as in TD)
    - **inaccurate** means $\statevalfun_\phi(\state) \neq \statevalfun_\pi(\state)$, for any reason
        - not yet trained, or trained toward targets that are still moving
        - a network too small to fit the values
        - a policy that has changed since the critic was fitted
- contrast with role 1, which never adds bias




# Step 4 of 4: both roles together, the TD error

**Why this section?** Use both roles at once. The result is one number per step: the critic's surprise.

> **Actor-critic weights each step by the TD error $\delta_\tt$: the critic's surprise. It compares two estimates of the return from $\stateseq_\tt$, one made after the action and one before.**

$$
\delta_\tt \;=\; \underbrace{\rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1})}_{\begin{array}{c}\text{after the action: one reward,}\\ \text{then the critic's value of where we landed}\end{array}} \;-\; \underbrace{\statevalfun_\phi(\stateseq_\tt)}_{\begin{array}{c}\text{before the action: what the}\\ \text{critic expected from this state}\end{array}}
\qquad \advseq_\tt \approx \delta_\tt
$$

- **left bracket, the bootstrap (role 2):** an estimate of the return **given the action $\actseq_\tt$ that was taken**
    - a one-sample estimate of $\actvalfun_\pi(\stateseq_\tt, \actseq_\tt)$
- **right bracket, the baseline (role 1):** the critic's estimate **before** the action
    - meant to be an average over the actions the policy might take
    - the critic is trained toward that average
- the difference is positive if the action beat the critic's expectation, and negative if it fell short
- **read $\delta_\tt$ as the surprise about the action taken, not as a new value for $\stateseq_\tt$**
    - the left bracket depends on the action, the right bracket on the state alone
    - the same number also trains the critic: its second job (below)


**Step 4 of 4, why it works: the TD error is a sample of the advantage**

> **$\delta_\tt$ is a one-sample estimate of the advantage $\actvalfun_\pi - \statevalfun_\pi$. That is why it can serve as the weight.**

- the exact advantage is $\actvalfun_\pi(\state, \act) - \statevalfun_\pi(\state)$ (B2)
- suppose first that the critic is exact: $\statevalfun_\phi = \statevalfun_\pi$
    - averaged over the next reward and the next state (the environment's randomness), given $\stateseq_\tt$ and $\actseq_\tt$, the left bracket is $\actvalfun_\pi$
        - $\Exp{\rewseq_{\tt+1},\, \stateseq_{\tt+1}}\bigl[\rewseq_{\tt+1} + \disc \, \statevalfun_\pi(\stateseq_{\tt+1}) \mid \stateseq_\tt, \actseq_\tt\bigr] = \actvalfun_\pi(\stateseq_\tt, \actseq_\tt)$
    - the right bracket is $\statevalfun_\pi(\stateseq_\tt)$, with nothing left to average
    - so $\Exp{\rewseq_{\tt+1},\, \stateseq_{\tt+1}}[\delta_\tt \mid \stateseq_\tt, \actseq_\tt] = \actvalfun_\pi - \statevalfun_\pi$: one sample of the advantage of the action taken
- with an inaccurate critic the sample is off
    - the gap is the bias of the bootstrap


**Step 4 of 4, using it: one actor-critic step**

- sample $\act \sim \pi_\theta(\cdot \mid \state)$
- observe $\rew$ and $\state'$
- $\delta \leftarrow \rew + \disc\,\statevalfun_\phi(\state') - \statevalfun_\phi(\state)$ (the critic's surprise; $\statevalfun_\phi(\text{terminal}) = 0$)
- critic: $\phi \leftarrow \phi + \alpha_\phi\,\delta\,\nabla_\phi \statevalfun_\phi(\state)$ (reduce its error; the target $\rew + \disc\,\statevalfun_\phi(\state')$ is held constant: a *semi-gradient* step)
- actor: $\theta \leftarrow \theta + \alpha\,\delta\,\nabla_\theta \log \pi_\theta(\act \mid \state)$ (weight by the surprise)
- $\alpha_\phi$ and $\alpha$ are two separate step sizes
- as in B2, the factor $\disc^\tt$ is dropped from the actor's update
- this is one update per time step
    - the ten runs below use batches of 8 episodes
- the same $\delta$ does two jobs: it moves the critic toward what happened, and it **is** the actor's weight
    - $\delta > 0$: the critic's value rises and the action becomes more likely
    - $\delta < 0$: the opposite


**After the four steps: what the critic buys and what it costs**

> **The critic buys much lower variance. An accurate critic adds almost no bias; an inaccurate one adds a bias nearly as large as the gradient.**

<img src="images/fl_critic_tradeoff.png" width=64%>

- FrozenLake, uniform policy
    - **variance** of a one-episode estimate, shown as its square root (the standard deviation)
    - **bias:** the distance of the average estimate from the exact gradient
- the first bar uses the exact state value as a baseline: a perfect critic in role 1
- the panels use different scales
- standard deviation
    - $0.45$ for the raw return (B2)
    - $0.36$ with the exact state value as baseline
    - $0.11$ with the TD error from an exact critic, $0.12$ from an inaccurate one
- bias: at the noise floor for the first two, $0.0095$ for the inaccurate critic, against an exact gradient of length $0.011$
    - "inaccurate" here is one case: the exact state value plus noise of standard deviation $0.05$ in every state
    - a bias near $0.002$ is the noise floor of the $40{,}000$ sampled episodes, about a fifth of the gradient's length

**Actor-critic in the unified form: what each part gives, and what it costs**

> **Compare with REINFORCE (B2): the weight and the baseline change, and both now come from the critic.**

<img src="images/b3_actor_critic_template.png" width=72%>

- orange: the two parts of the TD error $\delta_\tt$, the weight (slot 1) and the baseline (slot 3)
- blue: the rest of the template



**Actor-critic on FrozenLake: ten runs**

<img src="images/fl_actor_critic_curve.png" width=80%>

- the same batches of 8 episodes
- the critic is a table of values (a network with one-hot inputs), learned by TD(0)
- after 1,500 updates: mean return $0.15$ against $0.08$ for REINFORCE with a baseline; $J^* = 0.35$
- three of ten runs pass $0.25$, against two of ten for REINFORCE: better, but still noisy and uneven

**Why does actor-critic beat REINFORCE here?**

> **Mostly the bootstrap: a lower-variance weight makes steadier progress at the same step size. A state-dependent baseline alone helps less.**

- 30 runs each, same settings: mean return after 1,500 updates (standard error about $0.02$)
    - REINFORCE, moving baseline (B2): $0.06$ in this set of 30 runs ($0.08$ in the ten above), with 3 of 30 above $0.25$
    - REINFORCE, learned state-value baseline (role 1 only): $0.09$, 5 of 30
    - actor-critic, TD error (roles 1 and 2): $0.15$, 10 of 30
- the first gap is within noise; the second is about two standard errors
- the bootstrap also adds bias, but here it is outweighed by the variance it removes
- indicative, not a proof: one environment, one step size
- ten runs are few: the gap is suggestive, not established


**The same idea on a language model: prefixes are states, tokens are actions**

> **A completion is an episode. The actor picks each token, the critic values every prefix, and each token gets its own surprise $\delta$.**

<img src="images/b3_lm_actor_critic.png" width=64%>

- state: the prompt plus the tokens so far; action: the next token
- the reward often arrives only at the end: $0$ for every token but the last
- the critic is what gives each token its own credit
    - $\delta_\tt$ compares the value of the next prefix with the value of this one
- the cost: the critic must value every prefix of a long completion, a second network as large as the actor
- the full treatment is in C1


## The estimator dial: from the return to the TD error

**Why this section?** The advantage can be sampled for $p$ steps and then closed with the critic. The number of steps is the same *depth* dial as in sampling.

> **A dial from the return (no bias, high variance) to the TD error (critic bias, low variance).**

<img src="images/b3_dial.png" width=54%>

- the **$p$-step advantage**: $p$ sampled rewards, then the critic (so $p = 1$ is the TD error)
    - the critic's bias is discounted by $\disc^p$
    - the variance grows with $p$, since more random rewards are summed
- in a formula
$$
\hat{\advseq}^{(p)}_\tt = \rewseq_{\tt+1} + \dots + \disc^{p-1} \rewseq_{\tt+p} + \disc^p \, \statevalfun_\phi(\stateseq_{\tt+p}) - \statevalfun_\phi(\stateseq_\tt)
$$
- $p = 1$ is the TD error
- $p$ up to the end of the episode is the full return minus the critic's value
- the full return, with or without a baseline, bootstraps nothing: it carries no critic bias
    - the critic then acts only as a baseline (role 1)


**GAE: a dial between the TD error and the full return**

> **GAE weights the $p$-step advantages with a hyper-parameter $\lambda$ that acts as a dial.**

- there is a method, **GAE** (generalized advantage estimation), that combines all the $p$-step advantages
    - it weights them by a hyper-parameter $\lambda$, a number between $0$ and $1$
    - an exponentially weighted average: weight $(1 - \lambda)\lambda^{p-1}$ on the $p$-step advantage
- in a formula: a discounted sum of TD errors

$$
\hat{\advseq}_\tt = \sum_{l=0}^{\infty} (\disc \lambda)^l \, \delta_{\tt+l}
$$

- $\lambda$ is the dial
    - $\lambda = 0$: only $l = 0$ remains, so $\hat{\advseq}_\tt = \delta_\tt$: the TD error (lowest variance, critic bias)
    - $\lambda = 1$: the sum is the full sampled return minus the critic's value $\statevalfun_\phi(\stateseq_\tt)$ (highest variance, no bootstrap bias)
    - in between: a mixture of the two
- the dial trades bias for variance, the same trade as in sampling
    - a good setting is found by experiment
- PPO uses GAE as its advantage (B4)
- the details are in an **appendix**, `RL_B3_Appendix_GAE.ipynb`: worth reading before PPO


## The slot table: review, this session, preview

**Why this section?** B1 previewed three slots. Here is what is filled so far, what this session adds, and what the next sessions change.

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Slot 1: weight $\advseq$</div></th><th nowrap><div align="left">Slot 2: how far</div></th><th nowrap><div align="left">Slot 3: baseline source</div></th></tr></thead>
<tbody>
<tr><td colspan="4"><div align="left"><b>Review: filled so far</b></div></td></tr>
<tr><td nowrap><div align="left">REINFORCE, over-simplified (B1)</div></td><td nowrap><div align="left">return, no baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">none</div></td></tr>
<tr><td nowrap><div align="left">REINFORCE with baseline (B2)</div></td><td nowrap><div align="left">return minus baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">moving average of returns</div></td></tr>
<tr><td colspan="4"><div align="left"><b>This session</b></div></td></tr>
<tr><td nowrap><div align="left"><b>actor-critic</b></div></td><td nowrap><div align="left">TD error (or a $p$-step advantage, GAE)</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">learned critic $\statevalfun_\phi$</div></td></tr>
<tr><td colspan="4"><div align="left"><b>Preview (forward references, defined later): what the next sessions change</b></div></td></tr>
<tr><td nowrap><div align="left">PPO (B4)</div></td><td nowrap><div align="left">advantage from a critic (GAE)</div></td><td nowrap><div align="left"><b>clipped ratio</b></div></td><td nowrap><div align="left">learned critic</div></td></tr>
<tr><td nowrap><div align="left">GRPO (Part C)</div></td><td nowrap><div align="left">group-relative advantage</div></td><td nowrap><div align="left">clipped ratio</div></td><td nowrap><div align="left"><b>mean of a group of episodes</b></div></td></tr>
</tbody></table>

- in the preview rows, bold marks the cell that is new
- the preview rows use terms not yet defined: the clipped ratio is defined in B4, GRPO and its group baseline in Part C
    - read them only as a map of what is coming
- "step size $\alpha$" in slot 2 is the plain gradient step
- B4 asks how far one update may safely go


# What we established, and what it sets up

- **Established**
    - a **critic** is a second network $\statevalfun_\phi$: input a state, output the expected return, trained by regression
    - it plays two roles: a baseline (no bias) and a bootstrap (bias, lower variance)
    - **actor-critic** weights each step by the TD error, a sample of the advantage
    - the number of sampled steps before the critic is a dial between the return (no bias, high variance) and the TD error (bias, low variance)
    - GAE averages all the dial settings (appendix)

**What it sets up**

- **Sets up**
    - **the limitation:** the weight is better but still noisy, so each step is noisy
        - and the whole family so far is on-policy: after one update the batch came from the old policy, so it is discarded
        - a big step on a noisy estimate can also ruin the policy
    - next session: slot 2, how far one update may move: the surrogate objective and PPO, which also uses a critic and GAE
    - then, in Part C, GRPO: a baseline from a group of episodes with no critic
